In [334]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from weekly_team_predictions import *
from utils import *
pd.set_option('display.max_rows', None)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [335]:
schedules = build_schedules()
schedules.to_csv('schedules.csv', index=False)

In [336]:
features = assemble_features(schedules)
print("Number of features:", len(features))
print(features)

Number of features: 94
['spread_line', 'week', 'year', 'day_of_year', 'gametime', 'away_rest', 'home_rest', 'div_game', 'home_moneyline', 'away_moneyline', 'roof_closed', 'roof_dome', 'roof_open', 'roof_outdoors', 'surface_', 'surface_a_turf', 'surface_astroplay', 'surface_astroturf', 'surface_dessograss', 'surface_fieldturf', 'surface_grass', 'surface_grass ', 'surface_matrixturf', 'surface_sportturf', 'home_team_ARI', 'home_team_ATL', 'home_team_BAL', 'home_team_BUF', 'home_team_CAR', 'home_team_CHI', 'home_team_CIN', 'home_team_CLE', 'home_team_DAL', 'home_team_DEN', 'home_team_DET', 'home_team_GB', 'home_team_HOU', 'home_team_IND', 'home_team_JAX', 'home_team_KC', 'home_team_LA', 'home_team_LAC', 'home_team_LV', 'home_team_MIA', 'home_team_MIN', 'home_team_NE', 'home_team_NO', 'home_team_NYG', 'home_team_NYJ', 'home_team_OAK', 'home_team_PHI', 'home_team_PIT', 'home_team_SD', 'home_team_SEA', 'home_team_SF', 'home_team_STL', 'home_team_TB', 'home_team_TEN', 'home_team_WAS', 'away_t

In [337]:
#Train model 
def train_model(schedules: pd.DataFrame, features: list, seasons: list) -> tuple[LogisticRegression, StandardScaler, np.ndarray, pd.Series]:
    training_data = schedules.dropna(subset=features)
    training_data = training_data[training_data['year'].isin(seasons)]

    #Split into train and test sets
    X = training_data[features]
    y = training_data['result'] > 0 #Convert result from point dif to binary outcome 
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    #Scale features (put everything on scale 0-1)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Train Model
    model = LogisticRegression(max_iter=10000)
    model.fit(X_train_scaled, y_train)  
    return model, scaler, X_test_scaled, y_test

# Test model accuracy results
def accuracy_report(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))

In [ ]:
#build schedules
# assemble features
# train model
# get predictions
# build week1 and transform it
# convert week1 to account for weighted points based on outcome
SEASONS = list(range(2010, 2025))
model, scaler, X_test_scaled, y_test = train_model(schedules, features, seasons=SEASONS)
#Do prediction on test set
y_pred = model.predict(X_test_scaled)

In [339]:

WEEK = 2
YEAR = 2026

week_df, week_df_scaled = generate_inference_dataframe(WEEK, YEAR, schedules, scaler, features)
week_probs = generate_predictions(model, df_scaled=week_df_scaled)
print_predictions(week_df, week_probs)

Game: DET @ BUF, Predicted Win Probability for Home Team: 0.9026
Game: CAR @ ATL, Predicted Win Probability for Home Team: 0.7866
Game: NO @ BAL, Predicted Win Probability for Home Team: 0.8788
Game: MIN @ CHI, Predicted Win Probability for Home Team: 0.7903
Game: CIN @ HOU, Predicted Win Probability for Home Team: 0.8655
Game: PIT @ NE, Predicted Win Probability for Home Team: 0.6158
Game: GB @ NYJ, Predicted Win Probability for Home Team: 0.5769
Game: CLE @ TB, Predicted Win Probability for Home Team: 0.8695
Game: PHI @ TEN, Predicted Win Probability for Home Team: 0.6204
Game: JAX @ DEN, Predicted Win Probability for Home Team: 0.8083
Game: LV @ LAC, Predicted Win Probability for Home Team: 0.8217
Game: SEA @ ARI, Predicted Win Probability for Home Team: 0.5267
Game: WAS @ DAL, Predicted Win Probability for Home Team: 0.9564
Game: MIA @ SF, Predicted Win Probability for Home Team: 0.9283
Game: IND @ KC, Predicted Win Probability for Home Team: 0.8147
Game: NYG @ LA, Predicted Win Pr

In [340]:
results = generate_picks(week_df, week_probs)
print(results)

   favorite underdog  spread  favorite_win_prob  underdog_win_prob  \
0       BUF      DET     5.5           0.902555           0.097445   
1       CAR      ATL     2.5           0.213402           0.786598   
2       BAL       NO     8.5           0.878807           0.121193   
3       CHI      MIN     4.5           0.790271           0.209729   
4       HOU      CIN     3.0           0.865454           0.134546   
5        NE      PIT     4.5           0.615766           0.384234   
6        GB      NYJ     3.5           0.423100           0.576900   
7        TB      CLE     8.5           0.869511           0.130489   
8       PHI      TEN     7.0           0.379586           0.620414   
9       DEN      JAX     2.5           0.808346           0.191654   
10      LAC       LV     7.0           0.821691           0.178309   
11      SEA      ARI     3.5           0.473267           0.526733   
12      DAL      WAS     4.5           0.956448           0.043552   
13       SF      MIA

In [341]:
#Previous Year Comparison: get last years and figure out what score it would have made
YEAR = 2025
WEEK = None
previous_year, previous_year_scaled = generate_inference_dataframe(WEEK, YEAR, schedules, scaler, features)
predictions = generate_predictions(model, previous_year_scaled)
previous_year_results = generate_picks(previous_year, predictions)

previous_year_results['actual_winner'] = np.where(
    previous_year['result'] > 0,
    previous_year['home_team'],
    previous_year['away_team']
)

previous_year_points = np.where(
    previous_year_results['pick'] == previous_year_results['actual_winner'],
    np.where(
        previous_year_results['pick'] == previous_year_results['favorite'],
        1,
        np.where(
            previous_year_results['is_3pt_underdog'],
            3,
            2
        )
    ),
    0
)

print(f"Total Points for {YEAR}: {np.sum(previous_year_points)}")

#points if you picked perfectly (winners only)
perfect_points = np.where(
    previous_year_results['actual_winner'] == previous_year_results['favorite'],
    1,
    np.where(
        previous_year_results['is_3pt_underdog'],
        3,
        2
    )
)
print(f"Total Points if Picked Perfectly in {YEAR}: {np.sum(perfect_points)}")

previous_year_results.to_csv('2025.csv', index=False)

Total Points for 2025: 196
Total Points if Picked Perfectly in 2025: 383


In [342]:
importance = permutation_importance(
    model,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": importance["importances_mean"]
}).sort_values("importance", ascending=False)

print(feature_importance)

               feature    importance
0          spread_line  1.237209e-01
28       home_team_CAR  1.348837e-02
14            surface_  1.069767e-02
8       home_moneyline  6.511628e-03
44       home_team_MIN  4.651163e-03
53       home_team_SEA  4.651163e-03
29       home_team_CHI  4.651163e-03
13       roof_outdoors  4.651163e-03
37       home_team_IND  4.186047e-03
74        away_team_KC  4.186047e-03
66       away_team_CLE  3.720930e-03
17   surface_astroturf  3.720930e-03
54        home_team_SF  3.720930e-03
15      surface_a_turf  3.255814e-03
73       away_team_JAX  3.255814e-03
63       away_team_CAR  3.255814e-03
62       away_team_BUF  2.790698e-03
39        home_team_KC  2.790698e-03
38       home_team_JAX  2.325581e-03
10         roof_closed  2.325581e-03
36       home_team_HOU  1.860465e-03
27       home_team_BUF  1.395349e-03
68       away_team_DEN  9.302326e-04
79       away_team_MIN  9.302326e-04
41       home_team_LAC  9.302326e-04
11           roof_dome  4.651163e-04
2

In [343]:
#Test what features are contributing the most
def test_featureset_accuracy(features):
    model, _, X_test_scaled, y_test = train_model(schedules, features, seasons=SEASONS)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy: {accuracy:.4f}\n")
    return

In [344]:
team_features = [
    col for col in schedules.columns
    if col.startswith('home_team_') or col.startswith('away_team_')
]

roof_features = [
    col for col in schedules.columns
    if col.startswith('roof_')
]

surface_features = [
    col for col in schedules.columns
    if col.startswith('surface_')
]

# Regular numerical features
numeric_features = [
    'week',
    'year',
    'day_of_year',
    'gametime',
    'away_rest',
    'home_rest',
    'div_game',
]

vegas_features = [
    'spread_line',
    'home_moneyline',
    'away_moneyline'
]
#All Features
all_features = (numeric_features + vegas_features + roof_features + surface_features + team_features)
print("All Features")
test_featureset_accuracy(all_features)
#Everything Except Spread and Moneyline
non_vegas_features = (numeric_features + roof_features + surface_features + team_features)
print("Non-Vegas Features")
test_featureset_accuracy(non_vegas_features)
#Just Spread and Moneyline
just_vegas_features = (vegas_features)
print("Just Vegas Features")
test_featureset_accuracy(just_vegas_features)
#Everything but Team Data
non_team_features = (numeric_features + vegas_features + roof_features + surface_features)
print("Non-Team Features")
test_featureset_accuracy(non_team_features)
#Everything but Turf and Roof type data
non_env_features = (numeric_features + vegas_features + team_features)
print("Non-Roof and Surface Features")
test_featureset_accuracy(non_env_features)

All Features
Model Accuracy: 0.6140

Non-Vegas Features
Model Accuracy: 0.6047

Just Vegas Features
Model Accuracy: 0.6558

Non-Team Features
Model Accuracy: 0.6512

Non-Roof and Surface Features
Model Accuracy: 0.6140

